In [12]:
import mysql.connector
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# ==========================================
# 1. ESTABLISH INITIAL CONNECTION TO MYSQL
# ==========================================
# CHANGE 'YOUR_PASSWORD_HERE' to your actual local MySQL root password!
db_config = {
    'host': '127.0.0.1',
    'user': 'root',
    'password': 'Burki123@' 
}

print("⚡ Connecting to local MySQL Server...")
conn = mysql.connector.connect(**db_config)
cursor = conn.cursor()

# Create a clean database
cursor.execute("DROP DATABASE IF EXISTS telecom_portfolio;")
cursor.execute("CREATE DATABASE telecom_portfolio;")
cursor.execute("USE telecom_portfolio;")

print("📁 Database 'telecom_portfolio' created successfully.")

# ==========================================
# 2. CREATE SCHEMAS / TABLES
# ==========================================
print("🏗️ Creating structural schemas...")

cursor.execute('''
CREATE TABLE subscribers (
    subscriber_id VARCHAR(50) PRIMARY KEY,
    contract_type VARCHAR(50),
    tenure_months INT,
    monthly_charge DECIMAL(10,2),
    total_support_tickets INT,
    churned INT
);''')

cursor.execute('''
CREATE TABLE cell_towers (
    tower_id VARCHAR(50) PRIMARY KEY,
    latitude DECIMAL(10,6),
    longitude DECIMAL(10,6),
    capacity_tier VARCHAR(20)
);''')

cursor.execute('''
CREATE TABLE call_detail_records (
    record_id VARCHAR(50) PRIMARY KEY,
    subscriber_id VARCHAR(50),
    tower_id VARCHAR(50),
    service_type VARCHAR(20),
    timestamp DATETIME,
    duration_seconds INT,
    data_volume_mb DECIMAL(10,2),
    status VARCHAR(30)
);''')

# ==========================================
# 3. SIMULATE AND GENERATE REALISTIC DATA
# ==========================================
print("🎲 Generating telecom records...")
np.random.seed(42)
random.seed(42)

NUM_SUBSCRIBERS = 2000
NUM_TOWERS = 50
NUM_CDR_RECORDS = 20000

sub_ids = [f"SUB_{i:06d}" for i in range(1, NUM_SUBSCRIBERS + 1)]
tower_ids = [f"TWR_{i:04d}" for i in range(1, NUM_TOWERS + 1)]

# Subscribers Ingestion Data
sub_data = []
for s_id in sub_ids:
    sub_data.append((
        s_id,
        str(np.random.choice(['Prepaid', 'Month-to-Month', '2-Year Contract'], p=[0.5, 0.3, 0.2])),
        int(np.random.randint(1, 72)),
        float(np.random.uniform(15.0, 120.0)),
        int(np.random.poisson(0.8)),
        int(np.random.choice([0, 1], p=[0.85, 0.15]))
    ))

cursor.executemany("INSERT INTO subscribers VALUES (%s, %s, %s, %s, %s, %s)", sub_data)

# Cell Towers Ingestion Data
tower_data = []
for t_id in tower_ids:
    tower_data.append((
        t_id,
        float(np.random.uniform(33.5, 33.8)),
        float(np.random.uniform(72.9, 73.2)),
        str(np.random.choice(['Low', 'Medium', 'High'], p=[0.2, 0.5, 0.3]))
    ))

cursor.executemany("INSERT INTO cell_towers VALUES (%s, %s, %s, %s)", tower_data)

# CDR Ingestion Data
start_date = datetime(2026, 9, 1)
cdr_data = []
for i in range(1, NUM_CDR_RECORDS + 1):
    service = random.choice(['Voice', 'SMS', 'Data'])
    status = str(np.random.choice(['Success', 'Dropped', 'Failed_Congestion'], p=[0.92, 0.05, 0.03]))
    duration = random.randint(10, 600) if service == 'Voice' else 0
    data_vol = round(random.uniform(1.0, 150.0), 2) if service == 'Data' else 0.0
    timestamp = start_date + timedelta(seconds=random.randint(0, 30*24*60*60))
    
    cdr_data.append((
        f"CDR_{i:08d}",
        random.choice(sub_ids),
        random.choice(tower_ids),
        service,
        timestamp,
        duration,
        data_vol,
        status
    ))

cursor.executemany("INSERT INTO call_detail_records VALUES (%s, %s, %s, %s, %s, %s, %s, %s)", cdr_data)

conn.commit()
cursor.close()
conn.close()

print("🎉 Success! Your local MySQL database 'telecom_portfolio' is ready with 20,000 rows!")

⚡ Connecting to local MySQL Server...
📁 Database 'telecom_portfolio' created successfully.
🏗️ Creating structural schemas...
🎲 Generating telecom records...
🎉 Success! Your local MySQL database 'telecom_portfolio' is ready with 20,000 rows!
